In [ ]:
# ============================================================
# Cell 0: Setup + Load Model
# ============================================================
!pip install -q vllm==0.16.0 datasets 2>&1 | tail -3

import json, re, time, subprocess
from pathlib import Path
from typing import List
from datasets import load_dataset

TEMPERATURE = 0.6
TOP_P = 0.95
MAX_TOKENS = 16384
NUM_ROLLOUTS = 20
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
MODEL_SHORT = "deepseek-r1-distill-qwen-14b"
NUM_COTS = 10

BASE_DIR = Path("/content/gpqa_pilot")
BASE_DIR.mkdir(exist_ok=True)
ROLLOUTS_DIR = Path("/content/rollouts")


# --- Utility functions ---

def split_solution_into_chunks(solution_text: str) -> List[str]:
    if '<think>' in solution_text:
        solution_text = solution_text.split('<think>')[1].strip()
    if '</think>' in solution_text:
        solution_text = solution_text.split('</think>')[0].strip()
    chunks, current = [], ''
    i = 0
    while i < len(solution_text):
        current += solution_text[i]
        is_para = any(i+len(p)<=len(solution_text) and solution_text[i:i+len(p)]==p for p in ['\n\n','\r\n\r\n'])
        is_sent = i<len(solution_text)-1 and solution_text[i] in '.?!' and solution_text[i+1] in ' \n'
        if (is_para or is_sent) and current.strip():
            chunks.append(current.strip()); current = ''
        i += 1
    i = 0
    while i < len(chunks):
        if len(chunks[i]) < 10:
            if i == len(chunks)-1:
                if i > 0: chunks[i-1] += ' ' + chunks[i]; chunks.pop(i)
            else: chunks[i+1] = chunks[i] + ' ' + chunks[i+1]; chunks.pop(i)
            if i == 0 and len(chunks) == 1: break
        else: i += 1
    return chunks


def extract_mcq_answer(text: str) -> str:
    def find_choice(s):
        s = s.strip()
        for pat in [
            r'[Aa]nswer\s*(?:is|:)\s*\(?([A-D])\)?\.?',
            r'\b([A-D])\s*(?:is\s+correct|is\s+the\s+(?:correct|right)\s+answer)',
            r'\(([A-D])\)\s*$',
            r'\b([A-D])\b\s*$',
        ]:
            m = re.search(pat, s, re.MULTILINE)
            if m: return m.group(1).upper()
        return ''
    if '</think>' in text:
        ans = find_choice(text.split('</think>')[-1])
        if ans: return ans
    for line in reversed(text.strip().split('\n')[-5:]):
        ans = find_choice(line)
        if ans: return ans
    return ''


def extract_mcq_answer_strict(text: str) -> str:
    final = ''
    if '</think>' in text:
        final = extract_mcq_answer(text.split('</think>')[-1])
    body = text.split('</think>')[0] if '</think>' in text else text
    body_ans = ''
    for pat in [
        r'[Aa]nswer\s*(?:is|:)\s*\(?([A-D])\)?\.?',
        r'[Cc]orrect\s+(?:answer|option|choice)\s*(?:is|:)\s*\(?([A-D])\)?\.?',
        r'\b([A-D])\s*(?:is\s+correct|is\s+the\s+(?:correct|right))',
    ]:
        m = re.search(pat, body[-500:])
        if m: body_ans = m.group(1).upper(); break
    if final and body_ans:
        return final if final == body_ans else ''
    return final or body_ans


def check_mcq_answer(answer: str, gt: str) -> bool:
    return answer.strip().upper() == gt.strip().upper() if answer else False


def build_gpqa_prompt(problem: dict) -> str:
    q = problem['question']
    choices = ''
    for letter in ['A', 'B', 'C', 'D']:
        c = problem.get(f'choice_{letter}', problem.get(f'choice{letter}', ''))
        if c: choices += f'({letter}) {c}\n'
    return (
        f"Answer the following question step by step. "
        f"Choose the correct option (A, B, C, or D).\n\n"
        f"Question: {q}\n\n"
        f"{choices}\n"
        f"Answer:\n<think>\n"
    )


# --- Patch: 绕过 additional_chat_templates 404 ---
import transformers.utils.hub as _hub
_hub.list_repo_templates = lambda *a, **k: []

# --- Load vLLM ---
from vllm import LLM, SamplingParams
print('Loading model...')
llm = LLM(model=MODEL_NAME, dtype='bfloat16', max_model_len=16384, gpu_memory_utilization=0.9)
sp = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS)
print('Model loaded.')

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Loading model...
INFO 04-07 03:55:16 [utils.py:223] non-default args: {'dtype': 'bfloat16', 'max_model_len': 16384, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-14B'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 04-07 03:55:18 [model.py:529] Resolved architecture: Qwen2ForCausalLM
INFO 04-07 03:55:18 [model.py:1549] Using max model len 16384
INFO 04-07 03:55:18 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-07 03:55:18 [vllm.py:689] Asynchronous scheduling is enabled.
WARNING 04-07 03:55:20 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 04-07 03:56:29 [llm.py:355] Supported tasks: ['generate']
Model loaded.


In [ ]:
# ============================================================
# Cell 1: Download GPQA-Diamond + Screen ALL problems (198 × 10)
# ============================================================
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

import random
random.seed(42)

print('Downloading GPQA-Diamond...')
ds = load_dataset('Idavidrein/gpqa', 'gpqa_diamond', split='train')
print(f'Loaded {len(ds)} problems')

# Parse: shuffle choices into A/B/C/D randomly
problems = []
for i, item in enumerate(ds):
    q = item.get('Question', '')
    correct = item.get('Correct Answer', '')
    incorrects = [
        item.get('Incorrect Answer 1', ''),
        item.get('Incorrect Answer 2', ''),
        item.get('Incorrect Answer 3', ''),
    ]
    if not q or not correct:
        continue

    # Shuffle: put all 4 answers in random order
    options = [(correct, True)] + [(inc, False) for inc in incorrects if inc]
    random.shuffle(options)

    letters = 'ABCD'
    choices = {}
    gt_letter = ''
    for j, (text, is_correct) in enumerate(options):
        choices[letters[j]] = text
        if is_correct:
            gt_letter = letters[j]

    problems.append({
        'index': i, 'question': q,
        'choice_A': choices.get('A', ''), 'choice_B': choices.get('B', ''),
        'choice_C': choices.get('C', ''), 'choice_D': choices.get('D', ''),
        'gt_answer': gt_letter,
        'domain': item.get('High-level domain', 'unknown'),
        'subdomain': item.get('Subdomain', ''),
    })

print(f'Parsed {len(problems)} problems')
print(f'GT distribution: { {l: sum(1 for p in problems if p["gt_answer"]==l) for l in "ABCD"} }')

# --- Screen all ---
OUT_DIR = BASE_DIR / 'screening'
OUT_DIR.mkdir(exist_ok=True)

print(f'\nScreening ALL {len(problems)} problems × {NUM_COTS} CoTs = {len(problems)*NUM_COTS} total')

all_prompts, prompt_map = [], []
for ci, prob in enumerate(problems):
    prompt = build_gpqa_prompt(prob)
    for cot_i in range(NUM_COTS):
        all_prompts.append(prompt)
        prompt_map.append((ci, cot_i, prob, prompt))

t0 = time.time()
outputs = llm.generate(all_prompts, sp)
print(f'Batch done in {(time.time()-t0)/60:.1f} min')

for output, (ci, cot_i, prob, prompt) in zip(outputs, prompt_map):
    response = output.outputs[0].text
    answer = extract_mcq_answer_strict(response)
    is_correct = check_mcq_answer(answer, prob['gt_answer'])
    chunks = split_solution_into_chunks(response)
    cand_dir = OUT_DIR / f'candidate_{ci}'
    cand_dir.mkdir(exist_ok=True)
    json.dump(prob, open(cand_dir / 'problem.json', 'w'), indent=2, ensure_ascii=False)
    json.dump({
        'prompt': prompt, 'solution': response, 'full_cot': prompt + response,
        'answer': answer, 'gt_answer': prob['gt_answer'],
        'is_correct': is_correct, 'num_chunks': len(chunks),
    }, open(cand_dir / f'cot_{cot_i}.json', 'w'), indent=2, ensure_ascii=False)

# Summary
print(f'\n{"="*60}')
print('  GPQA-Diamond Screening — GOOD candidates (3-7/10 correct)')
print(f'{"="*60}')
good_count = 0
for ci, prob in enumerate(problems):
    cand_dir = OUT_DIR / f'candidate_{ci}'
    cots = [json.load(open(cand_dir / f'cot_{i}.json')) for i in range(NUM_COTS)]
    correct = [c for c in cots if c['is_correct']]
    incorrect = [c for c in cots if c.get('answer') and not c['is_correct']]
    unusable = [c for c in cots if not c.get('answer')]
    chunks = [c['num_chunks'] for c in cots]
    n_correct = len(correct)
    has_good = (3 <= n_correct <= 7 and
                any(c['num_chunks'] > 30 for c in correct) and
                any(c['num_chunks'] > 30 for c in incorrect))
    if has_good:
        gt = prob['gt_answer']
        dom = prob.get('domain', '?')[:20]
        print(f'  c{ci} (GT={gt}, {dom}): {n_correct}/10 ok, {len(incorrect)} wrong, {len(unusable)} unusable, chunks={chunks} ★')
        good_count += 1

print(f'\n{good_count} GOOD candidates out of {len(problems)} total')

In [ ]:
# ============================================================
# Cell 2: Download screening results
# ============================================================
!cd /content && zip -r gpqa_screening.zip gpqa_pilot/screening/
from google.colab import files
files.download('/content/gpqa_screening.zip')
print('Download screening results, then do the selection.')

In [ ]:
# ============================================================
# Cell 3 result reload
# This is another part of our experiment.
# Please enter the screnn results before running all the codes below.
# ============================================================

!cp /content/gpqa_20_candidates.json /content/
!cd /content && unzip -o gpqa_screening.zip
OUT_DIR = BASE_DIR / 'screening'

In [ ]:
# ============================================================
# Cell 4: After selection — verify all 20 chosen CoTs
# ============================================================
import json

OUT_DIR = BASE_DIR / 'screening'
with open("/content/gpqa_20_candidates.json") as f:
    candidates = json.load(f)

verified_pairs = []

for entry in candidates:
    cid = entry["candidate"]
    cc_idx = entry["correct_cot"]
    ic_idx = entry["incorrect_cot"]

    cand_dir = OUT_DIR / f"candidate_{cid}"
    prob = json.load(open(cand_dir / "problem.json"))
    correct_cot = json.load(open(cand_dir / f"cot_{cc_idx}.json"))
    incorrect_cot = json.load(open(cand_dir / f"cot_{ic_idx}.json"))

    print(f"\n=== Candidate {cid} [{entry['subdomain']}] ===")
    print(f"Problem: {prob['question'][:100]}...")
    print(f"GT: {prob['gt_answer']}")
    print(f"Correct CoT:   answer={correct_cot['answer']}, is_correct={correct_cot['is_correct']}, chunks={correct_cot['num_chunks']}")
    print(f"Incorrect CoT: answer={incorrect_cot['answer']}, is_correct={incorrect_cot['is_correct']}, chunks={incorrect_cot['num_chunks']}")

    for name, cot in [("correct", correct_cot), ("incorrect", incorrect_cot)]:
        sol = cot["solution"]
        body = sol.split("</think>")[0] if "</think>" in sol else sol
        after = sol.split("</think>")[-1] if "</think>" in sol else ""
        body_ans = extract_mcq_answer(body[-500:])
        final_ans = extract_mcq_answer(after)
        consistent = (body_ans == final_ans == cot["answer"]) if body_ans and final_ans else "partial"
        print(f"  {name}: body={body_ans}, after_think={final_ans}, field={cot['answer']}, consistent={consistent}")

    assert correct_cot["is_correct"] == True, f"Candidate {cid}: correct CoT is not correct!"
    assert incorrect_cot["is_correct"] == False, f"Candidate {cid}: incorrect CoT is not incorrect!"

    verified_pairs.append({
        "candidate": cid,
        "prob": prob,
        "correct_cot": correct_cot,
        "incorrect_cot": incorrect_cot,
        "subdomain": entry["subdomain"],
    })

print(f"\n✓ All {len(verified_pairs)} candidates verified.")

In [ ]:
# ============================================================
# Cell 5: Run Thought-Anchor rollouts (correct base solution)
#
#   NUM_ROLLOUTS  – number of resampled continuations per chunk
#   NUM_PROBLEMS  – how many of the 20 selected candidates to process
#                   (set to len(verified_pairs) for the full run)
#   SKIP_PROBLEMS – indices of problems whose rollouts are already
#                   saved on disk; they will be skipped to allow
#                   resuming after a runtime restart
# ============================================================
import subprocess
from google.colab import files

NUM_ROLLOUTS = 10
NUM_PROBLEMS = 10
SKIP_PROBLEMS = {0}

for idx, pair in enumerate(verified_pairs[:NUM_PROBLEMS]):
    if idx in SKIP_PROBLEMS:
        print(f"\n>>> Problem {idx} / Candidate {pair['candidate']} — SKIPPED (already done)")
        continue

    cid = pair["candidate"]
    prob = pair["prob"]
    cot = pair["correct_cot"]
    cot_type = "correct"

    print(f"\n{'='*60}")
    print(f"Problem {idx} / Candidate {cid} [{pair['subdomain']}]")
    print(f"{'='*60}")

    out = (ROLLOUTS_DIR / 'gpqa' / MODEL_SHORT
           / f'temperature_{TEMPERATURE}_top_p_{TOP_P}'
           / f'{cot_type}_base_solution' / f'problem_{idx}')
    out.mkdir(parents=True, exist_ok=True)

    base_prompt = build_gpqa_prompt(prob)
    sol_raw = cot['solution']
    full_cot = base_prompt + sol_raw

    sol_text = sol_raw
    if '<think>' in sol_text: sol_text = sol_text.split('<think>')[1].strip()
    if '</think>' in sol_text: sol_text = sol_text.split('</think>')[0].strip()

    chunks = split_solution_into_chunks(sol_raw)
    chunks_with_sentinel = chunks + [chunks[-1]]

    json.dump({'problem': prob['question'], 'gt_answer': prob['gt_answer'],
               'level': 'GPQA-Diamond', 'type': prob.get('domain', 'science'),
               'choice_A': prob['choice_A'], 'choice_B': prob['choice_B'],
               'choice_C': prob['choice_C'], 'choice_D': prob['choice_D']},
              open(out / 'problem.json', 'w'), indent=2)
    json.dump({'prompt': base_prompt, 'solution': sol_raw, 'full_cot': full_cot,
               'answer': cot['answer'], 'is_correct': cot['is_correct']},
              open(out / 'base_solution.json', 'w'), indent=2)
    json.dump({'source_text': full_cot, 'solution_text': sol_text, 'chunks': chunks_with_sentinel},
              open(out / 'chunks.json', 'w'), indent=2)

    cumulative, cur = [], ''
    for c in chunks: cur += c + ' '; cumulative.append(cur.strip())

    all_prompts, idx_map = [], []
    for ci, (chunk, prefix) in enumerate(zip(chunks, cumulative)):
        (out / f'chunk_{ci}').mkdir(exist_ok=True)
        sf = out / f'chunk_{ci}' / 'solutions.json'
        if sf.exists() and len(json.load(open(sf))) >= NUM_ROLLOUTS: continue
        pw = prefix.replace(chunk, '').strip()
        rp = base_prompt + pw
        for _ in range(NUM_ROLLOUTS):
            all_prompts.append(rp)
            idx_map.append((ci, rp, chunk, pw))

    N = len(chunks)
    chunk_N_dir = out / f'chunk_{N}'
    chunk_N_dir.mkdir(exist_ok=True)
    sf_N = chunk_N_dir / 'solutions.json'
    full_prefix = base_prompt + ' '.join(chunks)
    if not (sf_N.exists() and len(json.load(open(sf_N))) >= NUM_ROLLOUTS):
        for _ in range(NUM_ROLLOUTS):
            all_prompts.append(full_prefix)
            idx_map.append((N, full_prefix, chunks[-1], ' '.join(chunks)))

    print(f'  {N} chunks + sentinel, {len(all_prompts)} prompts')
    if not all_prompts:
        print('  Already done.')
    else:
        t0 = time.time()
        outputs = llm.generate(all_prompts, sp)
        print(f'  Batch done in {(time.time()-t0)/60:.1f} min')

        by_chunk = {}
        for o, (ci, rp, chunk, pw) in zip(outputs, idx_map):
            rtxt = o.outputs[0].text
            rc = split_solution_into_chunks(rtxt)
            ans = extract_mcq_answer(rtxt)
            ok = check_mcq_answer(ans, prob['gt_answer'])
            by_chunk.setdefault(ci, []).append({
                'chunk_removed': chunk, 'prefix_without_chunk': pw,
                'chunk_resampled': rc[0] if rc else '',
                'rollout': rtxt, 'full_cot': rp + rtxt,
                'answer': ans, 'is_correct': ok,
            })

        for ci, sols in sorted(by_chunk.items()):
            json.dump(sols, open(out / f'chunk_{ci}' / 'solutions.json', 'w'), indent=2)

        nc = sum(s['is_correct'] for ss in by_chunk.values() for s in ss)
        nt = sum(len(ss) for ss in by_chunk.values())
        print(f'  {len(by_chunk)} chunks, {nt} rollouts, {nc} correct ({nc/nt*100:.1f}%)')


    zip_name = f'gpqa_problem_{idx}.zip'
    zip_path = f'/content/{zip_name}'
    rel_path = str(out.relative_to('/content'))
    subprocess.run(['zip', '-r', zip_path, rel_path], cwd='/content', check=True)
    print(f'  Downloading {zip_name} ...')
    files.download(zip_path)

print("\n✓ All done!")


>>> Problem 0 / Candidate 7 — SKIPPED (already done)

Problem 1 / Candidate 127 [Molecular Biology]
  140 chunks + sentinel, 1410 prompts


Adding requests:   0%|          | 0/1410 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1410 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

  Batch done in 98.5 min
  141 chunks, 1410 rollouts, 743 correct (52.7%)


<IPython.core.display.Javascript object>

...

Processed prompts:   0%|          | 0/3310 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

  Batch done in 271.6 min
  331 chunks, 3310 rollouts, 1444 correct (43.6%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ All done!


In [ ]:
# ============================================================
# Cell 5: Verify & Download
# ============================================================
for idx, pair in enumerate(verified_pairs):
    cid = pair["candidate"]
    print(f"\n=== Problem {idx} / Candidate {cid} [{pair['subdomain']}] ===")
    for cot_type in ['correct', 'incorrect']:
        p = (ROLLOUTS_DIR / 'gpqa' / MODEL_SHORT
             / f'temperature_{TEMPERATURE}_top_p_{TOP_P}'
             / f'{cot_type}_base_solution' / f'problem_{idx}')
        if not (p / 'chunks.json').exists():
            print(f'  [{cot_type}] MISSING'); continue
        nc = len(json.load(open(p / 'chunks.json'))['chunks'])
        done = sum(1 for ci in range(nc) if (p / f'chunk_{ci}' / 'solutions.json').exists())
        total = sum(len(json.load(open(p / f'chunk_{ci}' / 'solutions.json')))
                    for ci in range(nc) if (p / f'chunk_{ci}' / 'solutions.json').exists())
        print(f'  [{cot_type}] {done}/{nc} chunks, {total} rollouts')

!cd /content && zip -r gpqa_rollouts.zip rollouts/gpqa/

from google.colab import files
files.download('/content/gpqa_rollouts.zip')
print('Done.')